In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Any
import json
import time

In [65]:
def plot_metrics(results: List[Dict[str, Any]], bacteria: str, timestamp: str = None):
    """
    Plots metrics for each combination of model and feature selector for a given bacteria.
    Saves a separate plot for each metric and each (model, feature selector, bacteria) combination.
    """
    import os
    print("Plotting metrics...")
    # Group results by (model, feature selector)
    grouped = {}
    for result in results:
        if result.get("bacteria") == bacteria:
            model_name = result.get("model_class", "UnknownModel")
            fs_name = result.get("feature_selector_class", "NoFS")
            key = (model_name, fs_name)
            if key not in grouped:
                grouped[key] = []
            grouped[key].append(result)


    y_min = 0
    y_max = 1

    for (model_name, fs_name), group_results in grouped.items():
        metrics_data = {
            "f1_score": [],
            "accuracy": [],
            "n_features": [],
            "scores": [], # Cross-Validation Scores
        }
        for result in group_results:
            metrics_data["n_features"].append(result.get("n_features_requested"))
            metrics_data["f1_score"].append(result.get("model_summary", {}).get("f1_score", 0))
            metrics_data["accuracy"].append(result.get("model_summary", {}).get("accuracy", 0))
            metrics_data["scores"].append(result.get("mean_cv_score", 0))

        feature_sizes = sorted(set(metrics_data["n_features"]))

        metrics_to_plot = {
            # "F1 Score": "f1_score",
            # "Accuracy": "accuracy",
            "Cross-Validation Score": "scores"
        }

        for metric_display_name, metric_key in metrics_to_plot.items():
            plt.figure(figsize=(9, 5))
            
            # Prepare y-values for the current metric
            y_values = [metrics_data[metric_key][metrics_data["n_features"].index(n)] for n in feature_sizes]

            plt.plot(
                feature_sizes,
                y_values,
                marker='o',
                label=metric_display_name,
                alpha=0.7,
                markersize=6
            )
            # If SuportVectorClassifier then model name is SVC
            if model_name == "SupportVectorMachineModel":
                model_name = "SVC"
            # If LogisticRegression then model name is LogisticRegression
            elif model_name == "LogisticRegressionModel":
                model_name = "LR"
            plt.title(f'{model_name} - {fs_name} - {metric_display_name}', fontsize=16, pad=10)
            plt.xlabel('Number of Features', fontsize=15)
            plt.ylabel(metric_display_name, fontsize=15)
            plt.ylim(y_min, y_max)
            plt.legend(fontsize=15)
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.xticks(fontsize=15)
            plt.yticks(fontsize=15)
            
            plot_timestamp = time.strftime("%Y%m%d-%H%M%S")
            os.makedirs("plots", exist_ok=True)
            # Sanitize metric_display_name for filename
            # plt.show()
            safe_metric_name = metric_display_name.replace(" ", "_")
            filename = f"./plots/{model_name}_{fs_name}_cv.jpg"
            plt.savefig(filename)
            plt.close()
            print(f"Metrics plot saved for {bacteria}, {model_name}, {fs_name} ({metric_display_name}) as {filename}")

In [66]:
import os
from typing import List, Dict, Any
import matplotlib.pyplot as plt
import time
import seaborn as sns

def plot_combined_metrics(results: List[Dict[str, Any]], bacteria: str, timestamp: str = None):
    """
    Plots accuracy and F1-score on the same plot for each combination of model and feature selector for a given bacteria.
    Saves a separate plot for each (model, feature selector, bacteria) combination with more contrasting colors.
    """
    print("Plotting combined metrics with more contrast...")
    # Group results by (model, feature selector)
    grouped = {}
    for result in results:
        if result.get("bacteria") == bacteria:
            model_name = result.get("model_class", "UnknownModel")
            fs_name = result.get("feature_selector_class", "NoFS")
            key = (model_name, fs_name)
            if key not in grouped:
                grouped[key] = []
            grouped[key].append(result)

    y_min = 0
    y_max = 1

    # Define more contrasting colors
    colors = ["#1f77b4", "#d62728"]  # Blue and Red

    for (model_name, fs_name), group_results in grouped.items():
        metrics_data = {
            "f1_score": [],
            "accuracy": [],
            "n_features": [],
        }
        for result in group_results:
            metrics_data["n_features"].append(result.get("n_features_requested"))
            metrics_data["f1_score"].append(result.get("model_summary", {}).get("f1_score", 0))
            metrics_data["accuracy"].append(result.get("model_summary", {}).get("accuracy", 0))

        feature_sizes = sorted(set(metrics_data["n_features"]))

        plt.figure(figsize=(9, 5))

        # Prepare y-values for accuracy and F1-score
        accuracy_values = [metrics_data["accuracy"][metrics_data["n_features"].index(n)] for n in feature_sizes]
        f1_score_values = [metrics_data["f1_score"][metrics_data["n_features"].index(n)] for n in feature_sizes]

        # Plot accuracy with a distinct color
        plt.plot(
            feature_sizes,
            accuracy_values,
            marker='s',
            label="Accuracy",
            alpha=0.7,
            color=colors[0],
            linewidth=2,
            markersize=8
        )

        # Plot F1-score with a different distinct color
        plt.plot(
            feature_sizes,
            f1_score_values,
            marker='o',
            label="F1 Score",
            alpha=0.7,
            color=colors[1],
            linewidth=2,
            markersize=6
        )
        # If SuportVectorClassifier then model name is SVC
        if model_name == "SupportVectorMachineModel":
            model_name = "SVC"
        # If LogisticRegression then model name is LogisticRegression
        elif model_name == "LogisticRegressionModel":
            model_name = "LR"

        plt.title(f'{model_name} - {fs_name} - Accuracy & F1 Score', fontsize=16, pad=10)
        plt.xlabel('Number of Features', fontsize=15)
        plt.ylabel('Metrics', fontsize=15)
        plt.ylim(y_min, y_max)
        plt.legend(fontsize=13)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.xticks(fontsize=15)
        plt.yticks(fontsize=15)

        plot_timestamp = time.strftime("%Y%m%d-%H%M%S")
        os.makedirs("plots", exist_ok=True)
        plt.tight_layout() # Adjust layout to prevent labels from overlapping
        # plt.show()
        filename = f"./plots/{model_name}_{fs_name}_cm.jpg"
        plt.savefig(filename)
        plt.close()
        print(f"Combined metrics plot saved for {bacteria}, {model_name}, {fs_name} as {filename}")

In [9]:
result_file = "genetic_run_results_kleb_20250508-144746.json"  # Path to your results file
with open(result_file, 'r') as file:
    results = json.load(file)



In [67]:
plot_combined_metrics(results, "kleb")

Plotting combined metrics with more contrast...
Combined metrics plot saved for kleb, SVC, PearsonCorrelationSelector as ./plots/SVC_PearsonCorrelationSelector_cm.jpg
Combined metrics plot saved for kleb, SVC, ShapFeatureSelector as ./plots/SVC_ShapFeatureSelector_cm.jpg
Combined metrics plot saved for kleb, SVC, RReliefF as ./plots/SVC_RReliefF_cm.jpg
Combined metrics plot saved for kleb, LR, PearsonCorrelationSelector as ./plots/LR_PearsonCorrelationSelector_cm.jpg
Combined metrics plot saved for kleb, LR, ShapFeatureSelector as ./plots/LR_ShapFeatureSelector_cm.jpg
Combined metrics plot saved for kleb, LR, RReliefF as ./plots/LR_RReliefF_cm.jpg


In [68]:
plot_metrics(results, "kleb")

Plotting metrics...
Metrics plot saved for kleb, SVC, PearsonCorrelationSelector (Cross-Validation Score) as ./plots/SVC_PearsonCorrelationSelector_cv.jpg
Metrics plot saved for kleb, SVC, ShapFeatureSelector (Cross-Validation Score) as ./plots/SVC_ShapFeatureSelector_cv.jpg
Metrics plot saved for kleb, SVC, RReliefF (Cross-Validation Score) as ./plots/SVC_RReliefF_cv.jpg
Metrics plot saved for kleb, LR, PearsonCorrelationSelector (Cross-Validation Score) as ./plots/LR_PearsonCorrelationSelector_cv.jpg
Metrics plot saved for kleb, LR, ShapFeatureSelector (Cross-Validation Score) as ./plots/LR_ShapFeatureSelector_cv.jpg
Metrics plot saved for kleb, LR, RReliefF (Cross-Validation Score) as ./plots/LR_RReliefF_cv.jpg
